# Module 2 - Topic 4: Introduction to Prompt Engineering
**Generative AI Fellowship — Beginner**

In this demo you will apply core prompt engineering techniques:
- Vague vs engineered prompts
- Zero-shot and few-shot prompting
- Role prompting
- Chain-of-thought prompting
- Output format control
- Prompt chaining


In [ ]:
# Cell 1 - Install
# !pip install openai python-dotenv

In [ ]:
# Cell 2 - Setup
# In Topic 3 we wrote out the full API call every time.
# Now we use a helper function called ask() to keep each cell short
# and focused on the prompt technique — not the API mechanics.

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def ask(prompt, system=None, temperature=0.7, max_tokens=400):
    """Send a prompt to GPT-4o-mini and return the text response."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print("Setup complete.")

In [ ]:
# Cell 3 - Vague vs Engineered Prompt
# The same task — but watch how much better the output is
# when we are specific about what we want

print("VAGUE PROMPT:")
print("-" * 50)
print(ask("Write something about my business"))

print("\nENGINEERED PROMPT:")
print("-" * 50)
print(ask(
    """Write a 3-sentence elevator pitch for a Nigerian B2B SaaS startup
that helps Lagos SMEs track inventory using WhatsApp.
Tone: professional and confident.
Include one specific benefit with a number or percentage."""
))

In [ ]:
# Cell 4 - Zero-Shot Prompting
# We give the model a task with no examples.
# It relies entirely on what it learned during training.

print(ask(
    """Classify the following customer complaint as: Billing, Delivery, or Product Quality.
Return only the category — no explanation.

Complaint: My order arrived three days late and the package was damaged.""",
    temperature=0
))

In [ ]:
# Cell 5 - Few-Shot Prompting
# Now we include examples to show the model exactly what we want.
# This is especially useful for domain-specific or locally relevant tasks.

# Example 1 — complaint classification with examples
print("Few-shot complaint classification:")
print(ask(
    """Classify customer complaints into: Billing, Delivery, or Product Quality.
Return only the category — no explanation.

Complaint: I was charged twice for the same order.
Category: Billing

Complaint: My package never arrived after 2 weeks.
Category: Delivery

Complaint: The customer service agent was very rude to me.
Category:""",
    temperature=0
))

print("\n" + "-" * 50)

# Example 2 — English to Pidgin translation
print("\nFew-shot English to Pidgin translation:")
print(ask(
    """Translate English to Nigerian Pidgin English.

English: The meeting has been postponed.
Pidgin: Dem don shift the meeting.

English: I need to make a payment.
Pidgin: I wan do payment.

English: The network is very slow today.
Pidgin:""",
    temperature=0.3
))

In [ ]:
# Cell 6 - Role Prompting
# Assigning a role in the system message changes the tone,
# vocabulary, and cultural focus of every response.

question = "How does inflation affect small businesses?"

print("NO ROLE:")
print("-" * 50)
print(ask(question, max_tokens=200))

print("\nCBN ECONOMIST ROLE:")
print("-" * 50)
print(ask(
    question,
    system="You are a senior economist at the Central Bank of Nigeria. Explain complex economic concepts in plain language that Nigerian small business owners in Lagos markets can understand. Always use Naira amounts and Nigerian examples.",
    max_tokens=200
))

In [ ]:
# Cell 7 - Chain-of-Thought Prompting
# Adding "Think through this step by step" forces the model
# to reason before answering — this dramatically improves accuracy
# on maths and multi-step problems.

problem = """A Lagos trader buys 50 bags of rice at ₦45,000 each and sells 35 bags
at ₦58,000 each. The remaining bags spoil and cannot be sold.
What is the total profit or loss?"""

print("WITHOUT chain-of-thought:")
print("-" * 50)
print(ask(problem, temperature=0))

print("\nWITH chain-of-thought:")
print("-" * 50)
print(ask(problem + "\n\nThink through this step by step before giving the final answer.", temperature=0))

print("\n--- The correct answer ---")
print("Cost:    50 × ₦45,000 = ₦2,250,000")
print("Revenue: 35 × ₦58,000 = ₦2,030,000")
print("Result:  ₦2,030,000 - ₦2,250,000 = -₦220,000 (Loss)")

In [ ]:
# Cell 8 - Output Format Control
# For production applications, we need consistent, parseable output.
# Always tell the model exactly what format you want.

question = "List the top 3 Nigerian banks by total assets."

print("UNCONTROLLED FORMAT:")
print("-" * 50)
print(ask(question))

print("\nJSON FORMAT:")
print("-" * 50)
print(ask(
    """List the top 3 Nigerian banks by total assets.
Return as JSON in this exact format:
{"banks": [{"rank": 1, "name": "..."}]}
Return only the JSON. No introduction. No explanation.""",
    temperature=0
))

In [ ]:
# Cell 9 - Prompt Chaining
# Complex tasks work better when broken into smaller steps.
# The output of Step 1 becomes the input to Step 2.

cv_text = """
Chidi Okafor | Lagos, Nigeria
Machine Learning Engineer — Flutterwave (2021–2024)
Data Scientist — Andela (2020–2021)
Skills: Python, TensorFlow, Scikit-learn, FastAPI
Education: B.Sc Computer Science, University of Lagos, 2019
"""

# Step 1 — Extract key information from the CV
step1_output = ask(
    f"""Extract from this CV:
- Candidate name
- Total years of experience (add up all roles)
- Top 3 technical skills

Return as a simple list. No extra explanation.

CV:
{cv_text}""",
    temperature=0
)

print("Step 1 — Extracted information:")
print(step1_output)

print("\n" + "-" * 50)

# Step 2 — Use Step 1 output to make a decision
step2_output = ask(
    f"""Based on this candidate profile:
{step1_output}

Does the candidate meet the minimum requirements for a Senior ML Engineer role?
Requirements: 5+ years total experience, Python required.

Answer with Yes or No and one sentence of reasoning.""",
    temperature=0
)

print("Step 2 — Screening decision:")
print(step2_output)